In [4]:
# CELL 1: Install
!pip -q install scikit-learn
import os, json, pickle
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import haversine_distances
print("OK")


OK


In [5]:
# CELL 2: Load AQI-36 files from PriSTI repo
os.chdir("/kaggle/working")
if not os.path.exists("PriSTI"):
    !git clone https://github.com/LMZZML/PriSTI.git

BASE   = "/kaggle/working/PriSTI/data/pm25/SampleData"
GROUND = BASE + "/pm25_ground.txt"
MISS   = BASE + "/pm25_missing.txt"
LATLNG = BASE + "/pm25_latlng.txt"
MEANSTD= "/kaggle/working/PriSTI/data/pm25/pm25_meanstd.pk"

# ground truth (no NaN)
X_ground = pd.read_csv(GROUND, index_col=0, parse_dates=True).values.astype(float)

# missing data (NaN for missing)
df_miss  = pd.read_csv(MISS,   index_col=0, parse_dates=True)
X        = df_miss.values.astype(float)

# GPS
df_latlng  = pd.read_csv(LATLNG)
latlng     = df_latlng[["latitude","longitude"]].values.astype(float)
sensor_ids = df_latlng["sensor_id"].astype(str).tolist()

# meanstd (pre-computed by PriSTI)
with open(MEANSTD, "rb") as f:
    train_mean, train_std = pickle.load(f)

T, N = X.shape
print("X (missing)  :", X.shape, "  NaN:", round(np.isnan(X).mean()*100, 2), "%")
print("X_ground     :", X_ground.shape)
print("latlng       :", latlng.shape)
print("train_mean   :", train_mean.shape, " range [%.2f, %.2f]" % (train_mean.min(), train_mean.max()))
print("train_std    :", train_std.shape,  " range [%.2f, %.2f]" % (train_std.min(),  train_std.max()))
print("sensor_ids   :", sensor_ids[:5], "...")


X (missing)  : (8759, 36)   NaN: 24.58 %
X_ground     : (8759, 36)
latlng       : (36, 2)
train_mean   : (36,)  range [60.82, 124.18]
train_std    : (36,)  range [62.33, 111.23]
sensor_ids   : ['1001', '1002', '1003', '1004', '1005'] ...


In [6]:
# CELL 3: Stage 1 - Node metadata
print("=" * 60)
print("STAGE 1 - PER-SENSOR STATISTICS")
print("=" * 60)

nodes_metadata = []

for j in range(N):
    col = X[:, j]           # may contain NaN
    obs = col[~np.isnan(col)]

    # missing rate
    missing_rate = float(np.isnan(col).mean())
    if   missing_rate < 0.05: missing_cat = "very_low"
    elif missing_rate < 0.10: missing_cat = "low"
    elif missing_rate < 0.20: missing_cat = "medium"
    elif missing_rate < 0.40: missing_cat = "high"
    else:                     missing_cat = "very_high"

    # signal mean/std from observed values (same as train_mean/std)
    sig_mean = float(train_mean[j])
    sig_std  = float(train_std[j])

    # autocorrelation lag-1 on observed values
    if len(obs) > 2:
        mu    = obs.mean()
        d     = obs - mu
        denom = (d**2).sum()
        autocorr = float((d[:-1]*d[1:]).sum() / denom) if denom > 0 else 0.
        autocorr = float(np.clip(autocorr, -1., 1.))
    else:
        autocorr = 0.

    # stability (variance of rolling variance, 24h window)
    win = 24
    roll_vars = []
    for k in range(0, T - win, win):
        chunk = col[k:k+win]
        chunk = chunk[~np.isnan(chunk)]
        if len(chunk) > win // 2:
            roll_vars.append(float(chunk.var()))
    if len(roll_vars) > 1:
        var_of_var = float(np.var(roll_vars))
        med_var    = float(np.median([v for v in roll_vars if v > 0])) if any(v>0 for v in roll_vars) else 1.
        stability  = "unstable" if var_of_var > med_var * 2 else "stable"
    else:
        var_of_var = 0.
        stability  = "stable"

    nodes_metadata.append({
        "sensor_id"        : str(sensor_ids[j]),
        "sensor_index"     : j,
        "latitude"         : round(float(latlng[j, 0]), 6),
        "longitude"        : round(float(latlng[j, 1]), 6),
        "missing_rate"     : round(missing_rate, 6),
        "missing_category" : missing_cat,
        "mean"             : round(sig_mean, 4),
        "std"              : round(sig_std,  4),
        "autocorr_lag1"    : round(autocorr, 6),
        "stability"        : stability,
        "temporal_var_mean": round(float(np.mean(roll_vars)) if roll_vars else 0., 4),
        "temporal_var_std" : round(float(np.std(roll_vars))  if roll_vars else 0., 4),
    })

print("Sensors processed:", len(nodes_metadata))
print("Missing rate  : min=%.1f%%  max=%.1f%%" % (
    min(n["missing_rate"] for n in nodes_metadata)*100,
    max(n["missing_rate"] for n in nodes_metadata)*100))
print("Autocorr lag1 : mean=%.3f" % np.mean([n["autocorr_lag1"] for n in nodes_metadata]))
print("mean range    : [%.2f, %.2f]" % (
    min(n["mean"] for n in nodes_metadata),
    max(n["mean"] for n in nodes_metadata)))
print("std  range    : [%.2f, %.2f]" % (
    min(n["std"]  for n in nodes_metadata),
    max(n["std"]  for n in nodes_metadata)))
print("\nSample node:")
print(json.dumps(nodes_metadata[0], indent=2))


STAGE 1 - PER-SENSOR STATISTICS
Sensors processed: 36
Missing rate  : min=12.1%  max=39.0%
Autocorr lag1 : mean=0.955
mean range    : [60.82, 124.18]
std  range    : [62.33, 111.23]

Sample node:
{
  "sensor_id": "1001",
  "sensor_index": 0,
  "latitude": 40.090679,
  "longitude": 116.173553,
  "missing_rate": 0.291243,
  "missing_category": "high",
  "mean": 88.0428,
  "std": 79.7105,
  "autocorr_lag1": 0.955071,
  "stability": "unstable",
  "temporal_var_mean": 1865.2658,
  "temporal_var_std": 3732.4923
}


In [7]:
# CELL 4: Stage 2 - Distance matrix + geographic adjacency (A_static)
print("=" * 60)
print("STAGE 2a - DISTANCE MATRIX + A_STATIC")
print("=" * 60)

# Haversine distance matrix (km)
latlon_rad  = np.radians(latlng)
dist_matrix = haversine_distances(latlon_rad) * 6371.0088

print("Distance matrix:", dist_matrix.shape)
print("Range (km): [%.2f, %.2f]" % (
    dist_matrix[dist_matrix>0].min(), dist_matrix.max()))

# Geographic adjacency — thresholded Gaussian kernel (same formula as PriSTI/GraphWaveNet)
finite_dist = dist_matrix.reshape(-1)
finite_dist = finite_dist[~np.isinf(finite_dist)]
finite_dist = finite_dist[finite_dist > 0]
sigma       = finite_dist.std()
adj         = np.exp(-np.square(dist_matrix / sigma))
adj[adj < 0.1] = 0.
np.fill_diagonal(adj, 0.)

print("\nA_static (geographic adjacency):")
print("  sigma =", round(sigma, 2), "km")
print("  Non-zero entries:", np.count_nonzero(adj), "/", N*N)
print("  Weight range: [%.4f, %.4f]" % (adj[adj>0].min(), adj.max()))


STAGE 2a - DISTANCE MATRIX + A_STATIC
Distance matrix: (36, 36)
Range (km): [1.62, 128.28]

A_static (geographic adjacency):
  sigma = 26.12 km
  Non-zero entries: 642 / 1296
  Weight range: [0.1012, 0.9962]


In [8]:
# CELL 5: Stage 2b - Pearson correlations -> relations_metadata
print("=" * 60)
print("STAGE 2b - PAIRWISE PEARSON CORRELATIONS")
print("=" * 60)

# AQI-36 uses NaN for missing — compute Pearson on co-observed values
# Use X_ground for training split (no NaN) for cleaner Pearson estimates
# Training months: 1,2,4,5,7,8,10,11 (from dataset_aqi36.py)
# But we don't have month-split here, so use all non-NaN pairs

MIN_CO_OBS  = 30
PEARSON_MIN = 0.15   # same threshold as W_sem formula

relations_metadata = []
n_pairs   = 0
n_skipped = 0

for i in range(N):
    for j in range(i+1, N):
        gw = float(adj[i, j])
        if gw == 0.:
            continue

        # co-observed mask (both non-NaN)
        mask_co = (~np.isnan(X[:, i])) & (~np.isnan(X[:, j]))
        n_co    = int(mask_co.sum())

        if n_co < MIN_CO_OBS:
            n_skipped += 1
            continue

        xi = X[mask_co, i]
        xj = X[mask_co, j]
        mu_i, mu_j = xi.mean(), xj.mean()
        si, sj     = xi.std(), xj.std()

        if si < 1e-8 or sj < 1e-8:
            pearson = 0.
        else:
            pearson = float(((xi - mu_i)*(xj - mu_j)).mean() / (si*sj))
            pearson = float(np.clip(pearson, -1., 1.))

        dist_km = float(dist_matrix[i, j])

        if   pearson > 0.85: rel_type = "STRONGLY_CORRELATED"
        elif pearson > 0.70: rel_type = "CORRELATED"
        else:                rel_type = "WEAKLY_CORRELATED"

        relations_metadata.append({
            "source"         : str(sensor_ids[i]),
            "target"         : str(sensor_ids[j]),
            "pearson"        : round(pearson, 6),
            "weight"         : round(pearson, 6),
            "distance_km"    : round(dist_km, 3),
            "gaussian_weight": round(gw, 6),
            "rel_type"       : rel_type,
        })
        n_pairs += 1

from collections import Counter
types    = Counter(r["rel_type"] for r in relations_metadata)
pearsons = [r["pearson"] for r in relations_metadata]
passing  = sum(1 for p in pearsons if p >= PEARSON_MIN)

print("Relations generated:", n_pairs, "(%d skipped)" % n_skipped)
print("  STRONGLY_CORRELATED (>0.85):", types["STRONGLY_CORRELATED"])
print("  CORRELATED (0.70-0.85)     :", types["CORRELATED"])
print("  WEAKLY_CORRELATED (<=0.70) :", types["WEAKLY_CORRELATED"])
print("  Pearson range: [%.3f, %.3f]  mean=%.3f" % (
    min(pearsons), max(pearsons), np.mean(pearsons)))
neg = sum(1 for p in pearsons if p < 0)
print("  Negative Pearson:", neg, "(will be filtered by delta=0.15)")
print("  Will pass delta=0.15:", passing, "->", passing*2, "directed edges")


STAGE 2b - PAIRWISE PEARSON CORRELATIONS
Relations generated: 321 (0 skipped)
  STRONGLY_CORRELATED (>0.85): 246
  CORRELATED (0.70-0.85)     : 75
  WEAKLY_CORRELATED (<=0.70) : 0
  Pearson range: [0.733, 0.979]  mean=0.887
  Negative Pearson: 0 (will be filtered by delta=0.15)
  Will pass delta=0.15: 321 -> 642 directed edges


In [9]:
# CELL 6: Save all files
OUTPUT_DIR = "/kaggle/working/aqi36_metadata"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# nodes_metadata.json
with open(OUTPUT_DIR + "/nodes_metadata.json", "w") as f:
    json.dump(nodes_metadata, f, indent=2)
print("nodes_metadata.json       -", len(nodes_metadata), "nodes")

# relations_metadata.json
with open(OUTPUT_DIR + "/relations_metadata.json", "w") as f:
    json.dump(relations_metadata, f, indent=2)
print("relations_metadata.json   -", len(relations_metadata), "relations")

# A_static.npy
np.save(OUTPUT_DIR + "/A_static.npy", adj.astype("float32"))
print("A_static.npy              -", adj.shape, " non-zero:", np.count_nonzero(adj))

# var_mean / var_std (signal mean/std for anomaly gate)
var_mean_arr = np.array([n["mean"] for n in nodes_metadata], dtype="float32")
var_std_arr  = np.array([n["std"]  for n in nodes_metadata], dtype="float32")
np.save(OUTPUT_DIR + "/var_mean.npy", var_mean_arr)
np.save(OUTPUT_DIR + "/var_std.npy",  var_std_arr)
print("var_mean.npy / var_std.npy - shape:", var_mean_arr.shape)

# NOTE: AQI-36 does NOT need pems_meanstd.pk or dist.npy in data/
# because dataset_aqi36.py reads pm25_meanstd.pk directly from data/pm25/
# and does NOT use a distance matrix (adjacency is inside generate_adj.py)
print("\nAll files saved to:", OUTPUT_DIR)
print("\nWhere these go in your repo:")
print("  neo4j_setup/metadata_aqi36/ <- all 5 files above")


nodes_metadata.json       - 36 nodes
relations_metadata.json   - 321 relations
A_static.npy              - (36, 36)  non-zero: 642
var_mean.npy / var_std.npy - shape: (36,)

All files saved to: /kaggle/working/aqi36_metadata

Where these go in your repo:
  neo4j_setup/metadata_aqi36/ <- all 5 files above


In [10]:
# CELL 7: Validation
print("=" * 60)
print("VALIDATION")
print("=" * 60)

print("\nA_static:")
print("  Shape      :", adj.shape)
print("  Symmetric  :", np.allclose(adj, adj.T))
print("  Diagonal=0 :", bool(np.all(np.diag(adj) == 0)))
print("  Non-zero   :", np.count_nonzero(adj))

print("\nW_sem preview:")
print("  Pairs passing delta=0.15 :", sum(1 for r in relations_metadata if r["pearson"] >= 0.15))
print("  Directed edges (x2)      :", sum(1 for r in relations_metadata if r["pearson"] >= 0.15)*2)

print("\nAnomaly thresholds - sample sensors:")
for i in range(3):
    n = nodes_metadata[i]
    lo = round(n["mean"] - 3*n["std"], 2)
    hi = round(n["mean"] + 3*n["std"], 2)
    print("  Sensor %d: mean=%.2f  std=%.2f  +-3sigma=[%.2f, %.2f]" % (
        i, n["mean"], n["std"], lo, hi))

print("\nValidation complete - AQI-36 metadata ready")


VALIDATION

A_static:
  Shape      : (36, 36)
  Symmetric  : True
  Diagonal=0 : True
  Non-zero   : 642

W_sem preview:
  Pairs passing delta=0.15 : 321
  Directed edges (x2)      : 642

Anomaly thresholds - sample sensors:
  Sensor 0: mean=88.04  std=79.71  +-3sigma=[-151.09, 327.17]
  Sensor 1: mean=77.93  std=74.60  +-3sigma=[-145.88, 301.74]
  Sensor 2: mean=88.57  std=77.44  +-3sigma=[-143.76, 320.89]

Validation complete - AQI-36 metadata ready


In [11]:
# CELL 8: Download
import shutil
shutil.make_archive("/kaggle/working/aqi36_metadata_all", "zip", OUTPUT_DIR)
print("Download: /kaggle/working/aqi36_metadata_all.zip")
print("\nContents:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(OUTPUT_DIR + "/" + fname)
    print("  %-35s %.1f KB" % (fname, size/1024))


Download: /kaggle/working/aqi36_metadata_all.zip

Contents:
  A_static.npy                        5.2 KB
  nodes_metadata.json                 12.1 KB
  relations_metadata.json             61.9 KB
  var_mean.npy                        0.3 KB
  var_std.npy                         0.3 KB
